# 🔬 Purple Agent — Plan-Execute vs Baseline Backtest

**Amaç:** Planner + Executor node eklemenin agent performansına etkisini ölçmek.

| | Mimari |
|---|---|
| **Baseline** | Sade ReAct — tek `create_agent` döngüsü, plan yok |
| **Candidate** | Plan-Execute — `planner node → executor node` (mevcut production) |

**Dataset:** `purple-agent-benchmark-v1`  
**Model:** `gpt-4o-mini` (her ikisi için sabit)  
**Metrikler:** `tool_usage_recall`, `task_success`

In [3]:
import os
import sys
import asyncio
import uuid
from dotenv import load_dotenv

# langgraph_agent module-level asyncio.run() Jupyter event loop'u öldürüyor
# Bu satır o bloğu devre dışı bırakır
os.environ["SKIP_AGENT_INIT"] = "true"

_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), "../../../"))
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

load_dotenv()

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "purple-agent-backtest")

MCP_ENDPOINT  = os.getenv("MCP_ENDPOINT", "http://localhost:8091").removesuffix("/mcp")
DATASET_NAME  = "purple-agent-benchmark-v1"
MODEL_NAME    = "local-qwen"   # LocalModel kullanılıyor

print(f"MCP Endpoint : {MCP_ENDPOINT}")
print(f"Dataset      : {DATASET_NAME}")
print(f"Model        : {MODEL_NAME}")
print(f"LangSmith    : {os.getenv('LANGSMITH_PROJECT')}")


MCP Endpoint : http://localhost:8091
Dataset      : purple-agent-benchmark-v1
Model        : local-qwen
LangSmith    : langchain-app


## 1️⃣ Agent Kurulumu

**Baseline:** Düz `create_agent` ReAct döngüsü — planner node yok, system prompt sabit.  
**Candidate:** `LangGraphAgent` — planner → executor graph (production implementasyonu).

In [4]:
import httpx
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ToolRetryMiddleware,
    wrap_tool_call,
    before_model,
)
from langchain_core.messages import ToolMessage

from src.purple_agent.langgraph_agent import (
    LangGraphAgent,
    MCPToolLoader,
    build_graph,
    _log_llm,
    _on_tool_error,
)

from custom_qwen import LocalModel

# ── Shared: MCP tool loader ──────────────────────────────────────────────────
_tool_loader = MCPToolLoader(MCP_ENDPOINT)

# ── Baseline: sade ReAct, plan yok ──────────────────────────────────────────
_BASELINE_SYSTEM_PROMPT = """\
You are a capable AI assistant. You have access to tools.
Use them to complete the task accurately and efficiently.
Always provide ALL required arguments when calling a tool.
On error, adjust arguments and retry.
"""

def build_baseline_graph(tools):
    """Düz create_agent — planner node yok, tek ReAct döngüsü."""
    # model'e tools verilmez — create_agent içinde bind_tools(tools) otomatik çağrılır
    return create_agent(
        model=LocalModel(),
        tools=tools,
        system_prompt=_BASELINE_SYSTEM_PROMPT,
        middleware=[
            _log_llm,
            ToolRetryMiddleware(
                max_retries=3, backoff_factor=2.0,
                initial_delay=1.0, max_delay=30.0, jitter=True,
                retry_on=(ConnectionError, TimeoutError, httpx.HTTPError),
                on_failure="continue",
            ),
            _on_tool_error,
        ],
    )


# ── Candidate: Plan-Execute (production build_graph) ────────────────────────
def build_candidate_graph(tools):
    """Plan-Execute graph — planner node → executor node."""
    return build_graph(model=LocalModel(), tools=tools)


# ── Lazy-initialized state ───────────────────────────────────────────────────
_shared_tools = None        # MCP tools (her iki agent paylaşır)
_baseline_graph = None      # Baseline ReAct graph
_candidate_graph_cache = None  # Candidate Plan-Execute graph

print("✅ Agent kurulumu tamamlandı")


✅ Agent kurulumu tamamlandı (modül reload edildi)


## 2️⃣ Target Functions & Evaluators

Her iki ajan için ayrı `target` fonksiyonu. Evaluator'lar paylaşımlı.

In [5]:
from langchain_core.messages import HumanMessage
from langsmith import Client
from langsmith.schemas import Example, Run

# ── Shared tools loader ──────────────────────────────────────────────────────
async def _ensure_tools() -> list:
    global _shared_tools
    if _shared_tools is None:
        _shared_tools = await _tool_loader.load()
        print(f"✅ MCP tools yüklendi ({len(_shared_tools)} tool)")
    return _shared_tools


# ── Baseline: sade ReAct graph cache ────────────────────────────────────────
async def _ensure_baseline():
    global _baseline_graph
    if _baseline_graph is None:
        tools = await _ensure_tools()
        _baseline_graph = build_baseline_graph(tools)
        print("✅ Baseline graph hazır")
    return _baseline_graph


# ── Candidate: Plan-Execute graph cache ─────────────────────────────────────
async def _ensure_candidate():
    global _candidate_graph_cache
    if _candidate_graph_cache is None:
        tools = await _ensure_tools()
        _candidate_graph_cache = build_candidate_graph(tools)
        print("✅ Candidate graph hazır")
    return _candidate_graph_cache


# ── Helpers ──────────────────────────────────────────────────────────────────
def _extract_question(inputs: dict) -> str:
    for key in ("text", "question", "input"):
        if inputs.get(key):
            return inputs[key]
    return next((v for v in inputs.values() if isinstance(v, str)), "")


# ── Target Functions ─────────────────────────────────────────────────────────
async def target_baseline(inputs: dict) -> dict:
    graph = await _ensure_baseline()
    question = _extract_question(inputs)
    if not question:
        return {"error": "No question found", "messages": []}
    try:
        config = {"configurable": {"thread_id": str(uuid.uuid4())}}
        return await graph.ainvoke(
            {"messages": [HumanMessage(content=question)]},
            config=config,
        )
    except Exception as e:
        return {"error": str(e), "messages": []}


async def target_candidate(inputs: dict) -> dict:
    graph = await _ensure_candidate()
    question = _extract_question(inputs)
    if not question:
        return {"error": "No question found", "messages": []}
    try:
        config = {"configurable": {"thread_id": str(uuid.uuid4())}}
        return await graph.ainvoke(
            {"messages": [HumanMessage(content=question)], "plan": []},
            config=config,
        )
    except Exception as e:
        return {"error": str(e), "messages": []}


# ── Evaluators ───────────────────────────────────────────────────────────────
def tool_usage_recall(run: Run, example: Example) -> dict:
    """Beklenen toolların kaçının çağrıldığını ölçer (recall)."""
    expected: list = (example.outputs or {}).get("expected_tools", [])
    if not expected:
        return {"key": "tool_usage_recall", "score": None}

    messages = (run.outputs or {}).get("messages", [])
    called = set()
    for msg in messages:
        tool_calls = (
            getattr(msg, "tool_calls", None)
            or (msg.get("tool_calls") if isinstance(msg, dict) else None)
            or (msg.get("kwargs", {}).get("tool_calls") if isinstance(msg, dict) else None)
            or []
        )
        for tc in tool_calls:
            name = tc.get("name") if isinstance(tc, dict) else getattr(tc, "name", None)
            if name:
                called.add(name)

    missing = [t for t in expected if t not in called]
    score   = (len(expected) - len(missing)) / len(expected)
    return {
        "key": "tool_usage_recall",
        "score": round(score, 3),
        "comment": f"expected={expected} | called={sorted(called)} | missing={missing}",
    }


def task_success(run: Run, example: Example) -> dict:
    """Agent hata döndürmeden tamamladıysa 1, aksi hâlde 0."""
    outputs = run.outputs or {}
    has_error = bool(outputs.get("error"))
    messages  = outputs.get("messages", [])
    has_answer = any(
        getattr(m, "content", None) or (isinstance(m, dict) and m.get("content"))
        for m in messages[-2:] if not getattr(m, "tool_calls", None)
    )
    score = 0 if has_error else (1 if has_answer else 0)
    return {"key": "task_success", "score": score}


print("✅ Target fonksiyonları ve evaluator'lar hazır")


✅ Target fonksiyonları ve evaluator'lar hazır
   'task' key kullanılıyor (input yerine — create_agent çakışması önlendi)


## 3️⃣ Baseline Experiment — Düz ReAct (Plan yok)

`create_agent` ile tek döngü. Planner node yok, sistem promptu sabit.

In [ ]:
from langsmith import aevaluate

_run_id = str(uuid.uuid4())[:6]
BASELINE_PREFIX  = f"baseline-react-{MODEL_NAME}-{_run_id}"

print(f"🚀 Baseline başlıyor: {BASELINE_PREFIX}")
print(f"   Mimari : Düz ReAct (planner node YOK)\n")

baseline_results = await aevaluate(
    target_baseline,
    data=DATASET_NAME,
    evaluators=[tool_usage_recall, task_success],
    experiment_prefix=BASELINE_PREFIX,
    max_concurrency=1,
    description="Baseline: Düz create_agent ReAct döngüsü, plan adımı yok",
    metadata={"model": MODEL_NAME, "architecture": "react", "variant": "baseline"},
)

print(f"\n✅ Baseline tamamlandı → {baseline_results.experiment_name}")

## 4️⃣ Candidate Experiment — Plan-Execute Graph

`planner node → executor node`. Planner önce adım listesi üretir, executor uygular.

In [11]:
from langsmith import aevaluate

CANDIDATE_PREFIX = f"candidate-plan-execute-{MODEL_NAME}-"

print(f"🚀 Candidate başlıyor: {CANDIDATE_PREFIX}")
print(f"   Mimari : Plan-Execute (planner node → executor node)\n")

candidate_results = await aevaluate(
    target_candidate,
    data=DATASET_NAME,
    evaluators=[tool_usage_recall, task_success],
    experiment_prefix=CANDIDATE_PREFIX,
    max_concurrency=1,
    description="Candidate: Plan-Execute — planner node → executor node",
    metadata={"model": MODEL_NAME, "architecture": "plan-execute", "variant": "candidate"},
)

print(f"\n✅ Candidate tamamlandı → {candidate_results.experiment_name}")


🚀 Candidate başlıyor: candidate-plan-execute-local-qwen-
   Mimari : Plan-Execute (planner node → executor node)

View the evaluation results for experiment: 'candidate-plan-execute-local-qwen--9cfbc2f3' at:
https://smith.langchain.com/o/378a72ed-098a-4298-9931-7e1696a11886/datasets/b45ac299-b3c4-415e-8e09-c508d5d02fa4/compare?selectedSessions=91216ba5-111f-4af7-8136-865a219a7d73




0it [00:00, ?it/s]

[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 0
  tool_choice: not set
[LocalModel] Response received:
  content: ["search for 'Quarterly Review' document in Google Drive", "getGoogleDocContent to retrieve the cont...
  tool_calls: []
  finish_reason: stop
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 76
  tool_choice: auto
[LocalModel] Response received:
  content: (none)...
  tool_calls: [{'id': 'chatcmpl-tool-eca8c0bdfaaf45d3b0b533aa034a773e', 'type': 'function', 'function': {'name': 'search', 'arguments': '{"query": "name=\'Quarterly Review\' and mimeType=\'application/vnd.google-apps.document\'", "pageSize": 50, "pageToken": null}'}}]
  finish_reason: tool_calls
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 76
  tool_choice: auto
[LocalModel] Response received:
  content: (none)...
  tool_calls: [{'id': 'chatcmpl-tool-7adda2

1it [00:56, 56.28s/it]

[LocalModel] Response received:
  content: I'm encountering persistent technical issues with executing the required functions due to an asynchr...
  tool_calls: []
  finish_reason: stop
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 0
  tool_choice: not set
[LocalModel] Response received:
  content: ["google_search for YouTube video 'MKBHD iPhone Review'", "get_video_info from the search result to ...
  tool_calls: []
  finish_reason: stop
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 76
  tool_choice: auto
[LocalModel] Response received:
  content: (none)...
  tool_calls: [{'id': 'chatcmpl-tool-dfb3253dba1546849818497936e133da', 'type': 'function', 'function': {'name': 'google_search', 'arguments': '{"q": "MKBHD iPhone Review", "num": 1, "tbs": "", "page": 1, "autocorrect": true}'}}]
  finish_reason: tool_calls
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-In

...

[LocalModel] Response received:
  content: I'm encountering a technical issue with executing the search function asynchronously. Let me try to ...
  tool_calls: []
  finish_reason: stop
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 0
  tool_choice: not set
[LocalModel] Response received:
  content: ["google_search for weather in London today", "scrape the top weather result to check if it's rainin...
  tool_calls: []
  finish_reason: stop
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 76
  tool_choice: auto
[LocalModel] Response received:
  content: (none)...
  tool_calls: [{'id': 'chatcmpl-tool-7c6de7c34cdc4ebcaa867862f3ea1337', 'type': 'function', 'function': {'name': 'google_search', 'arguments': '{"q": "weather in London today", "num": 10, "tbs": "", "page": 1, "autocorrect": true}'}}]
  finish_reason: tool_calls
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A2

4it [02:24, 33.77s/it]

[LocalModel] Response received:
  content: It seems there is a technical limitation preventing the execution of the `google_search` tool in thi...
  tool_calls: []
  finish_reason: stop
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 0
  tool_choice: not set


4it [02:28, 37.17s/it]


CancelledError: 

## 5️⃣ Sonuç Karşılaştırması

İki experiment'ın metriklerini yan yana karşılaştır.

In [ ]:
def _summarize(results, label: str) -> dict:
    rows = list(results)
    if not rows:
        return {"label": label, "n": 0}

    def _avg(key: str) -> float:
        scores = [
            r.get("feedback", {}).get(key)
            for r in rows
            if r.get("feedback", {}).get(key) is not None
        ]
        return round(sum(scores) / len(scores), 3) if scores else float("nan")

    return {
        "label":             label,
        "n":                 len(rows),
        "tool_usage_recall": _avg("tool_usage_recall"),
        "task_success":      _avg("task_success"),
    }


b = _summarize(baseline_results,  "Baseline  (ReAct, plan yok)")
c = _summarize(candidate_results, "Candidate (Plan-Execute)")

header = f"{'Metric':<25} {'Baseline':>14} {'Candidate':>14} {'Delta':>10}"
sep    = "─" * len(header)
print(sep)
print(header)
print(sep)

for m in ["tool_usage_recall", "task_success"]:
    bv = b.get(m, float("nan"))
    cv = c.get(m, float("nan"))
    try:
        delta  = f"{cv - bv:+.3f}"
        winner = "✅" if cv >= bv else "⚠️"
    except TypeError:
        delta, winner = "N/A", ""
    print(f"{m:<25} {bv:>14} {cv:>14} {delta:>10}  {winner}")

print(sep)
print(f"\n📌 Örnek sayısı : {b['n']}")
print(f"? Baseline     : {BASELINE_PREFIX}")
print(f"📊 Candidate    : {CANDIDATE_PREFIX}")
print(f"🔗 LangSmith    : https://smith.langchain.com")

NameError: name 'baseline_results' is not defined

Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /runs/multipart (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 8] nodename nor servname provided, or not known)"))'))
Content-Length: 535833
API Key: lsv2_********************************************7ftrace=019c7626-50b8-7861-9dea-506a55570cad,id=019c7640-5b2b-7851-9f36-8ae463cf885f; trace=019c7626-50b8-7861-9dea-506a55570cad,id=019c7640-5a90-75d1-8310-354ef0f0cb4d; trace=019c7626-50b8-7861-9dea-506a55570cad,id=019c7640-7cf8-7ee2-ab3b-a597d614cbe2; trace=019c7626-50b8-7861-9dea-506a55570cad,id=019c7640-7cf8-7ee2-ab3b-a597d614cbe2; trace=019c7626-50b8-7861-9dea-506a55570cad,id=

: 

: 

: 